<a href="https://colab.research.google.com/github/adnan57575757/End-to-End-Sentimental-analysis/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Imports & Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import bz2
import os
import json

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── TensorFlow / Keras ──────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Embedding, GRU, LSTM, Bidirectional,
    Dense, Dropout, BatchNormalization, SpatialDropout1D,
    GlobalMaxPooling1D, GlobalAveragePooling1D,
    Conv1D, MaxPooling1D, Concatenate, Input
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# ── Sklearn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import time

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

# 2. Loading Data

In [ ]:
def load_fasttext(file_path):
    # reads the fasttext bz2 file and converts it to dataframe
    texts = []
    labels = []

    f = bz2.open(file_path, 'rt')   # opening compressed file

    for line in f:
        # checking label manually
        if line.startswith('__label__2'):
            labels.append(1)
        else:
            labels.append(0)

        # removing label part from text
        clean_line = line.replace('__label__1 ', '')
        clean_line = clean_line.replace('__label__2 ', '')
        clean_line = clean_line.strip()

        texts.append(clean_line)

    f.close()   # probably important to close it

    data = pd.DataFrame({'text': texts, 'label': labels})
    return data


train_path = '/kaggle/input/datasets/bittlingmayer/amazonreviews/train.ft.txt.bz2'
test_path  = '/kaggle/input/datasets/bittlingmayer/amazonreviews/test.ft.txt.bz2'

train_df = load_fasttext(train_path)   # big file, takes time
test_df = load_fasttext(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain labels:")
print(train_df['label'].value_counts())

print("\nTest labels:")
print(test_df['label'].value_counts())

# just checking imbalance, might need later

# 3. Exploratory Data Analysis — Review Length

In [ ]:
# checking how long the reviews are (in words)
lengths = []

for text in train_df['text']:
    split_words = text.split()
    lengths.append(len(split_words))

print("Min:", np.min(lengths))
print("Max:", np.max(lengths))

mean_len = np.mean(lengths)
print("Mean:", round(mean_len, 2))

print("Median:", np.median(lengths))

# checking percentiles to decide max length later
p90 = np.percentile(lengths, 90)
p95 = np.percentile(lengths, 95)
p99 = np.percentile(lengths, 99)

print("90th %:", p90)
print("95th %:", p95)
print("99th %:", p99)

# might use ~150 or something based on this

In [ ]:
# trying to balance dataset (same number of pos and neg)
train_ml = train_df.groupby('label').apply(
    lambda x: x.sample(500000, random_state=42)
)

train_ml = train_ml.reset_index(drop=True)   # resetting index after sampling

X_train_ml = train_ml['text']
y_train_ml = train_ml['label']

# using full test set (not touching it during training)
X_test_ml = test_df['text']
y_test_ml = test_df['label']


# tf-idf vectorizer (for converting text to numbers)
tfidf = TfidfVectorizer(
    max_features=50000,
    stop_words='english',
    ngram_range=(1,2),   # using uni + bi grams
    min_df=5,
    max_df=0.9
)

print("Train size:", len(X_train_ml), "| Test size:", len(X_test_ml))

# not sure if 50k features is overkill but trying anyway

## 4.1 Model 1 — Logistic Regression

In [ ]:
# ── Pipeline: TF-IDF → Logistic Regression ───────────────────
# Pipeline chains steps so fit() and predict() apply both automatically
pipeline_lgr = Pipeline([
    ('tfidf', tfidf), # Step 1: vectorise raw text
    ('clf',   LogisticRegression(
        max_iter=1000,  # Allow enough iterations for saga to converge
        solver='saga',  # 'saga' supports L1/L2 and scales to large datasets
        n_jobs=-1       # Use all CPU cores in parallel
    ))
])

start = time.time()
pipeline_lgr.fit(X_train_ml, y_train_ml)  # Trains vectoriser + classifier together
print(f'LGR Training time : {time.time()-start:.1f}s')

# ── Evaluate ─────────────────────────────────────────────────
pred_lgr = pipeline_lgr.predict(X_test_ml)
prob_lgr = pipeline_lgr.predict_proba(X_test_ml)[:, 1]

print(f'LGR Accuracy : {accuracy_score(y_test_ml, pred_lgr):.4f}')
print(f'LGR ROC-AUC  : {roc_auc_score(y_test_ml, prob_lgr):.4f}')
# Per-class precision, recall and F1
print(classification_report(y_test_ml, pred_lgr, digits=4))

## 4.2 Model 2 — Linear SVM

In [ ]:
# ── Pipeline: TF-IDF → Linear SVM ────────────────────────────
# LinearSVC is faster than SVC(kernel='linear') for large text datasets
# It optimises a hinge loss rather than logistic loss

pipeline_svm = Pipeline([
    ('tfidf', tfidf),
    ('clf',   LinearSVC())  # Default: C=1.0, max_iter=1000
])

start = time.time()
pipeline_svm.fit(X_train_ml, y_train_ml)
print(f'SVM Training time : {time.time()-start:.1f}s')

# ── Evaluate ─────────────────────────────────────────────────
# Note: LinearSVC does NOT support predict_proba → no ROC-AUC here

pred_svm = pipeline_svm.predict(X_test_ml)
print(f'SVM Accuracy : {accuracy_score(y_test_ml, pred_svm):.4f}')
print(classification_report(y_test_ml, pred_svm, digits=4))

## 4.3 Quick Prediction Demo — Logistic Regression

In [ ]:
# Manual smoke-test: check whether the LGR pipeline gives intuitive predictions
# on three hand-crafted sentences covering positive, negative, and neutral tones

sample_texts = [
    'Amazing product! Love it!',
    'Terrible, broke after 2 days.',
    'It is okay, nothing special.'
]
print('Quick prediction check:')
for t in sample_texts:
    p = pipeline_lgr.predict([t])[0]
    print(f"  {'Positive 🥰' if p else 'Negative 😤'}  →  '{t}'")

# HYPERPARAMETERS

In [ ]:
# ── All training hyperparameters defined in one place ─────────
# Changing a value here propagates to every model automatically

VOCAB_SIZE    = 50_000   # Maximum vocabulary: keep the 50K most frequent tokens
                         # (original was 20K — more vocab = better coverage)

MAX_LEN       = 150      # Pad/truncate every review to exactly 150 tokens
                         # 95th-percentile review length is 161 words, so this
                         # captures the vast majority of each review's content
                         # (original was 100 — too short, lost key context)

EMBED_DIM     = 128      # Each token is represented as a 128-dimensional float vector
                         # (original was 64 — richer representations improve accuracy)

EPOCHS        = 6        # Maximum training epochs; EarlyStopping usually stops earlier

BATCH_SIZE    = 1024     # Samples processed per gradient update
                         # CRITICAL FIX: original value was 8, causing 315K steps/epoch
                         # and making training effectively impossible to complete

DL_SAMPLES    = 1_200_000  # Stratified sample from the 3.6M train_df
                            # 600K Positive + 600K Negative → perfectly balanced

VAL_SPLIT     = 0.15     # 15% of DL_SAMPLES → 180K validation rows

DROPOUT       = 0.3      # 30% of Dense layer neurons dropped during training
SPATIAL_DROP  = 0.2      # 20% of embedding feature channels dropped per step

LEARNING_RATE = 3e-4     # Adam learning rate (3×10⁻⁴ — slightly conservative
                         # for stable convergence with large batches)

SAVE_DIR      = '/kaggle/working/models'  # Output directory for saved models

os.makedirs(SAVE_DIR, exist_ok=True)   # Create directory if it doesn't exist

# ── Print a summary with derived stats ───────────────────────
print(f'[Hyperparameters]')
print(f'  vocab_size   = {VOCAB_SIZE:,}')
print(f'  max_len      = {MAX_LEN}')
print(f'  embed_dim    = {EMBED_DIM}')
print(f'  batch_size   = {BATCH_SIZE}')
print(f'  dl_samples   = {DL_SAMPLES:,}')
print(f'  val_split    = {VAL_SPLIT}')
train_n = int(DL_SAMPLES * (1 - VAL_SPLIT))
# steps_per_epoch = train_samples / batch_size
print(f'  train rows   = {train_n:,}  →  steps/epoch ≈ {train_n // BATCH_SIZE:,}')
print(f'  save_dir     = {SAVE_DIR}')


## 5.2 Preprocessing

In [ ]:
# ── Step 1: Draw a stratified sample from train_df ───────────
# We use 1.2M instead of all 3.6M for two reasons:
#   1. Tokenising 3.6M texts is very slow (~5 min) and memory-heavy
#   2. 1.2M is sufficient signal for the DL models to reach >94% accuracy
# groupby('label') + sample(600K) guarantees exactly 600K per class
print('Sampling from train_df ...')
dl_train = train_df.groupby('label', group_keys=False).apply(
    lambda g: g.sample(DL_SAMPLES // 2, random_state=42)
).reset_index(drop=True)

print(f'DL sample shape : {dl_train.shape}')
print(dl_train['label'].value_counts())

In [ ]:
# ── Step 2: Train / Validation split ─────────────────────────
# test_df is NEVER split or sampled here — it is reserved entirely for final evaluation.
# stratify=labels ensures the 85/15 split keeps the 50/50 class balance intact
X_raw_train, X_raw_val, y_train, y_val = train_test_split(
    dl_train['text'].values,    # Raw review strings
    dl_train['label'].values,   # Binary labels
    test_size=VAL_SPLIT,        # 15% → X_val
    random_state=42,            # Reproducible split
    stratify=dl_train['label'].values   # Preserve class ratio
)

print(f'Train texts : {len(X_raw_train):,}')
print(f'Val   texts : {len(X_raw_val):,}')
print(f'Test  texts : {len(test_df):,}  ← official test_df (never touched)')

In [ ]:
# ── Step 3: Fit the Tokeniser on training texts ONLY ─────────
# Fitting on test data would constitute data leakage.
# oov_token='<OOV>' maps unseen words to a reserved index (1)
# instead of silently dropping them, which helps the model generalise.
print('Fitting tokeniser ...')
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_raw_train)   # Builds vocabulary from training corpus only

print(f'Full vocab size  : {len(tokenizer.word_index):,}')   # All unique words seen
print(f'Capped at        : {VOCAB_SIZE:,}')                  # Only top 50K used


In [ ]:
# ── Step 4: Encode text → integer sequences, then pad ────────
# texts_to_sequences replaces each word with its integer index.
# pad_sequences brings all sequences to exactly MAX_LEN tokens:
#   - Shorter reviews get zeros appended at the end (padding='post')
#   - Longer reviews are cut from the end (truncating='post')
print('Encoding sequences ...')

X_train = pad_sequences(
    tokenizer.texts_to_sequences(X_raw_train),
    maxlen=MAX_LEN, padding='post', truncating='post'
)
X_val = pad_sequences(
    tokenizer.texts_to_sequences(X_raw_val),
    maxlen=MAX_LEN, padding='post', truncating='post'
)

# Apply the SAME tokeniser (fitted on train only) to encode test_df
# This is the only correct way — the test set must not influence preprocessing
X_test = pad_sequences(
    tokenizer.texts_to_sequences(test_df['text'].values),
    maxlen=MAX_LEN, padding='post', truncating='post'
)
y_test = test_df['label'].values

print(f'X_train : {X_train.shape}')   # (1_020_000, 150)
print(f'X_val   : {X_val.shape}')     # (  180_000, 150)
print(f'X_test  : {X_test.shape}')    # (  400_000, 150) ← from test_df


## 5.3 Model Definitions

In [ ]:
def _emb():

    return [
        Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN, name='embedding'),
        SpatialDropout1D(SPATIAL_DROP, name='spatial_drop'),
    ]


## BiLSTM Model

In [ ]:
def build_bilstm():

    model = Sequential([
        *_emb(),
        # First BiLSTM: 128 units each direction → 256 output features per timestep
        # dropout=0.2 drops input connections; recurrent_dropout=0.1 drops recurrent ones
        Bidirectional(LSTM(128, return_sequences=True,
                           dropout=0.2, recurrent_dropout=0.1), name='bilstm_1'),
        # Second BiLSTM: refines the sequence representation at lower dimensionality
        Bidirectional(LSTM(64,  return_sequences=True,
                           dropout=0.2, recurrent_dropout=0.1), name='bilstm_2'),
        # Collapse (timesteps, features) → (features,) by taking max across time
        GlobalMaxPooling1D(name='global_max_pool'),
        # Classification head
        Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),  # L2 penalises large weights
        BatchNormalization(),   # Normalise activations → stable gradients
        Dropout(DROPOUT),
        Dense(64, activation='relu'),
        Dropout(DROPOUT / 2),
        Dense(1, activation='sigmoid', name='output'),  # P(Positive)
    ], name='BiLSTM')

    model.compile(
        optimizer=Adam(LEARNING_RATE),
        loss='binary_crossentropy',  # Standard loss for binary classification
        metrics=['accuracy']
    )
    return model

build_bilstm().summary()


## 5.5 BiGRU Model

In [ ]:
def build_bigru():

    model = Sequential([
        *_emb(),
        # First BiGRU: 128 units each direction → 256 features per timestep
        Bidirectional(GRU(128, return_sequences=True,
                          dropout=0.2, recurrent_dropout=0.1), name='bigru_1'),
        # Second BiGRU: compresses to 64 units each direction
        Bidirectional(GRU(64,  return_sequences=True,
                          dropout=0.2, recurrent_dropout=0.1), name='bigru_2'),
        GlobalMaxPooling1D(name='global_max_pool'),
        Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(DROPOUT),
        Dense(64, activation='relu'),
        Dropout(DROPOUT / 2),
        Dense(1, activation='sigmoid', name='output'),
    ], name='BiGRU')

    model.compile(optimizer=Adam(LEARNING_RATE),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

build_bigru().summary()


## 5.6 Conv1D + BiLSTM Hybrid Model

In [ ]:
def build_conv_bilstm():

    inp = Input(shape=(MAX_LEN,), name='input')
    x   = Embedding(VOCAB_SIZE, EMBED_DIM, name='embedding')(inp)
    x   = SpatialDropout1D(SPATIAL_DROP)(x)

    # ── CNN branch: parallel convolutions for different n-gram sizes ──
    c3 = Conv1D(128, 3, activation='relu', padding='same')(x)  # Trigram patterns
    c5 = Conv1D(128, 5, activation='relu', padding='same')(x)  # 5-gram patterns
    x  = Concatenate()([c3, c5])   # Shape: (batch, MAX_LEN, 256)
    x  = MaxPooling1D(2)(x)        # Shape: (batch, 75, 256) — halve the timesteps
    x  = BatchNormalization()(x)   # Normalise after CNN
    x  = Dropout(0.2)(x)

    # ── RNN branch: model sequential context in compressed representation ──
    x   = Bidirectional(LSTM(128, return_sequences=True,
                              dropout=0.2, recurrent_dropout=0.1))(x)

    # ── Dual pooling: capture both max signal and average tone ────────
    avg = GlobalAveragePooling1D()(x)   # Mean across all timesteps
    mx  = GlobalMaxPooling1D()(x)       # Max across all timesteps
    x   = Concatenate()([avg, mx])      # Shape: (batch, 512)

    # ── Classification head ───────────────────────────────────────────
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x   = BatchNormalization()(x)
    x   = Dropout(DROPOUT)(x)
    out = Dense(1, activation='sigmoid', name='output')(x)

    model = Model(inp, out, name='Conv1D_BiLSTM')
    model.compile(optimizer=Adam(LEARNING_RATE),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

build_conv_bilstm().summary()


## 5.7 Callbacks

In [ ]:
def get_callbacks(model_name):

    ckpt_path = os.path.join(SAVE_DIR, f'{model_name}_best.keras')
    return [
        EarlyStopping(
            monitor='val_accuracy',
            patience=4,                  # Wait 4 epochs before stopping
            restore_best_weights=True,   # Roll back to best epoch weights on stop
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,       # Multiply LR by 0.5 on plateau
            patience=2,       # Plateau = no improvement for 2 epochs
            min_lr=1e-6,      # Never go below this LR
            verbose=1
        ),
        ModelCheckpoint(
            filepath=ckpt_path,
            monitor='val_accuracy',
            save_best_only=True,   # Only save when val_accuracy improves
            verbose=1
        ),
    ]


## 5.8 Train All Models

In [ ]:
# ── Define which models to train ─────────────────────────────
# Each key is the model name; each value is the builder function.
# Calling builder() creates a fresh, untrained model instance.
models_config = {
    'LSTM'         : build_bilstm,
    'GRU'          : build_bigru,
    'Conv1D_BiLSTM': build_conv_bilstm,
}

# Shared storage for results across all models
histories   = {}   # Training history (loss/accuracy per epoch)
results     = {}   # Final test-set metrics (acc, auc, cm, report)
predictions = {}   # Predicted probabilities and hard labels for test_df

for name, builder in models_config.items():
    print(f'\n{"="*60}')
    print(f'  Training : {name}')
    print(f'{"="*60}')

    # Build a fresh model for each iteration (no weight sharing between models)
    model = builder()

    # ── Training ─────────────────────────────────────────────
    # validation_data is used to monitor val_accuracy and val_loss each epoch
    # but NEVER updates the model weights — it is purely for monitoring
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks(name),
        verbose=1
    )
    histories[name] = history

    # ── Evaluation on the official test_df (400K rows) ───────
    # Use double batch_size for inference — no gradients stored, so memory allows it
    y_pred_prob = model.predict(
        X_test, batch_size=BATCH_SIZE * 2, verbose=0
    ).flatten()

    # Convert probabilities to binary labels using 0.5 threshold
    y_pred = (y_pred_prob >= 0.5).astype(int)

    # Compute all evaluation metrics
    acc    = accuracy_score(y_test, y_pred)
    auc    = roc_auc_score(y_test, y_pred_prob)  # Needs probabilities, not hard labels
    cm     = confusion_matrix(y_test, y_pred)
    report = classification_report(
        y_test, y_pred,
        target_names=['Negative', 'Positive'],
        digits=4
    )

    # Store for plotting and comparison later
    results[name]     = {'acc': acc, 'auc': auc, 'cm': cm, 'report': report}
    predictions[name] = (y_pred_prob, y_pred)

    print(f'\n  Test Accuracy (test_df) : {acc:.4f}')
    print(f'  ROC-AUC                 : {auc:.4f}')
    print(f'\n{report}')


---
## 5.9 Save the Best Model

In [ ]:
# ── Step 1: Identify the best model by test accuracy ─────────
# max() with a key function finds the model_name with the highest 'acc' value
best_name = max(results, key=lambda k: results[k]['acc'])
best_acc  = results[best_name]['acc']
best_auc  = results[best_name]['auc']

print(f'Best model  : {best_name}')
print(f'Accuracy    : {best_acc:.4f}')
print(f'ROC-AUC     : {best_auc:.4f}')

# ── Step 2: Reload the checkpoint saved by ModelCheckpoint ───
# ModelCheckpoint saved the epoch with the highest val_accuracy during training.
# We reload it here to ensure we use the best generalising weights,
# not the weights from the final epoch (which may have slightly overfit).
best_ckpt  = os.path.join(SAVE_DIR, f'{best_name}_best.keras')
best_model = tf.keras.models.load_model(best_ckpt)
print(f'\nLoaded checkpoint : {best_ckpt}')

# ── Step 3: Save as the canonical final model ─────────────────
# This is the single file used by the deployment API
final_path = os.path.join(SAVE_DIR, 'best_model_final.keras')
best_model.save(final_path)
print(f'Final model saved : {final_path}')

# ── Step 4: Save the tokeniser ───────────────────────────────
# The tokeniser vocabulary must be saved alongside the model.
# Without it, we cannot convert new text into the same integer sequences
# the model was trained on — predictions would be completely wrong.
tok_path = os.path.join(SAVE_DIR, 'tokenizer_config.json')
with open(tok_path, 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())
print(f'Tokeniser saved   : {tok_path}')

# ── Step 5: Save metadata for the deployment API ─────────────
# The FastAPI backend reads this file to know MAX_LEN, model name, accuracy etc.
# without having to load the full model just for those values.
meta = {
    'best_model_name' : best_name,
    'test_accuracy'   : round(best_acc, 6),
    'test_roc_auc'    : round(best_auc, 6),
    'vocab_size'      : VOCAB_SIZE,
    'max_len'         : MAX_LEN,
    'embed_dim'       : EMBED_DIM,
    'batch_size'      : BATCH_SIZE,
    'dl_samples'      : DL_SAMPLES,
    'all_results'     : {k: {'acc': round(v['acc'],6),
                              'auc': round(v['auc'],6)}
                          for k, v in results.items()},
}
meta_path = os.path.join(SAVE_DIR, 'model_meta.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Metadata saved    : {meta_path}')

# ── Step 6: Print a manifest of everything saved ──────────────
print('\n' + '='*55)
print('  Saved files:')
print('='*55)
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath    = os.path.join(SAVE_DIR, fname)
    size_mb  = os.path.getsize(fpath) / 1e6
    print(f'  {fname:<38}  {size_mb:6.1f} MB')


---
# 6. Results & Evaluation

In [ ]:
# ── Unified results table: ML baselines + DL models ──────────
print('\n' + '='*62)
print('  FINAL RESULTS  —  evaluated on test_df (400,000 rows)')
print('='*62)
print(f'  {"Model":<24}  {"Accuracy":>10}  {"ROC-AUC":>10}')
print('  ' + '-'*48)

# ML baselines (evaluated on full test_df)
print(f'  {"LGR (ML)":<24}  '
      f'{accuracy_score(y_test_ml, pred_lgr):>10.4f}  '
      f'{roc_auc_score(y_test_ml, prob_lgr):>10.4f}')
print(f'  {"SVM (ML)":<24}  '
      f'{accuracy_score(y_test_ml, pred_svm):>10.4f}  {"N/A":>10}')

# DL models (evaluated on full test_df)
for model_name, r in results.items():
    tag = '  ← BEST ✅' if model_name == best_name else ''
    print(f'  {model_name:<24}  {r["acc"]:>10.4f}  {r["auc"]:>10.4f}{tag}')

print('='*62)
print(f'\nBest model saved to : {final_path}')


## 6.1 Training Curves

In [ ]:
# ── Plot accuracy and loss curves for all DL models ──────────
# Two sub-plots per model: left = accuracy, right = loss
# Comparing train vs val curves reveals:
#   - Overfitting: train >> val (gap widens over epochs)
#   - Underfitting: both curves plateau at a low value
#   - Good fit: train ≈ val, both curves converge smoothly
n = len(histories)
fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))
if n == 1:
    axes = [axes]   # Ensure axes is always a list of rows

for ax_row, (name, history) in zip(axes, histories.items()):
    h  = history.history
    ep = range(1, len(h['accuracy']) + 1)

    # Accuracy subplot
    ax_row[0].plot(ep, h['accuracy'],     label='Train')
    ax_row[0].plot(ep, h['val_accuracy'], label='Val')
    ax_row[0].set_title(f'{name} — Accuracy')
    ax_row[0].set_xlabel('Epoch')
    ax_row[0].legend()
    ax_row[0].grid(True, alpha=0.3)

    # Loss subplot
    ax_row[1].plot(ep, h['loss'],     label='Train')
    ax_row[1].plot(ep, h['val_loss'], label='Val')
    ax_row[1].set_title(f'{name} — Loss')
    ax_row[1].set_xlabel('Epoch')
    ax_row[1].legend()
    ax_row[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()


## 6.2 Confusion Matrices

In [ ]:
# ── Confusion matrix for each DL model ───────────────────────
# Rows = True labels, Columns = Predicted labels
#   Top-left  (TN): Negative reviews correctly classified as Negative
#   Top-right (FP): Negative reviews incorrectly classified as Positive
#   Bot-left  (FN): Positive reviews incorrectly classified as Negative
#   Bot-right (TP): Positive reviews correctly classified as Positive
# Ideally the diagonal (TN + TP) dominates and off-diagonal (FP + FN) is small.
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 5))
if len(results) == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    sns.heatmap(r['cm'], annot=True, fmt=',d', cmap='Blues', ax=ax,
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    tag = ' ✅' if name == best_name else ''
    ax.set_title(f'{name}{tag}\nAcc={r["acc"]:.4f}  AUC={r["auc"]:.4f}')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()


## 6.3 ROC Curves

In [ ]:
# ── ROC (Receiver Operating Characteristic) curve ────────────
# Each curve shows the trade-off between:
#   True Positive Rate  (Recall / Sensitivity) — how many Positives we catch
#   False Positive Rate (1 - Specificity)       — how many Negatives we wrongly flag
#
# A perfect model hugs the top-left corner (TPR=1, FPR=0).
# The diagonal dashed line represents a random (50/50) classifier.
# AUC (Area Under Curve) summarises this in one number:
#   AUC = 1.0 → perfect, AUC = 0.5 → random, AUC < 0.5 → worse than random
plt.figure(figsize=(8, 6))

# ML baseline: Logistic Regression (LGR)
fpr_lgr, tpr_lgr, _ = roc_curve(y_test_ml, prob_lgr)
plt.plot(fpr_lgr, tpr_lgr, linestyle='--',
         label=f'LGR  AUC={roc_auc_score(y_test_ml, prob_lgr):.4f}')

# DL models
for name, (prob, _) in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    tag = ' ✅' if name == best_name else ''
    plt.plot(fpr, tpr, label=f'{name}{tag}  AUC={results[name]["auc"]:.4f}')

# Reference line: random classifier
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models  (test_df, 400K rows)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'roc_curves.png'), dpi=150)
plt.show()


---
## 7. Load Saved Model & Run Inference

In [ ]:
# ── This cell can be run in any future session ────────────────
# It reloads the best model and tokeniser from disk without retraining.
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf, json, os

SAVE_DIR = '/kaggle/working/models'

# ── Load metadata ─────────────────────────────────────────────
# Contains model name, accuracy, hyperparameters — no need to hardcode them
with open(os.path.join(SAVE_DIR, 'model_meta.json')) as f:
    meta = json.load(f)
print('Best model  :', meta['best_model_name'])
print('Accuracy    :', meta['test_accuracy'])
print('ROC-AUC     :', meta['test_roc_auc'])

# ── Load tokeniser ────────────────────────────────────────────
# tokenizer_from_json reconstructs the exact vocabulary and settings
# that were used during training
with open(os.path.join(SAVE_DIR, 'tokenizer_config.json'), 'r') as f:
    loaded_tokenizer = tokenizer_from_json(f.read())

# ── Load model ────────────────────────────────────────────────
# Keras .keras format stores both architecture and weights in one file
loaded_model = tf.keras.models.load_model(
    os.path.join(SAVE_DIR, 'best_model_final.keras')
)
print('Model loaded successfully!')


def predict_sentiment(texts):
    """
    Predict sentiment for a list of review strings.

    Pipeline:
        raw text → integer sequences → padded array → model → probabilities → labels

    Args:
        texts (list[str]): One or more review strings.

    Prints:
        For each text: probability score, label (Positive/Negative), and the text.
    """
    # Encode using the saved tokeniser (same vocabulary as training)
    seqs = loaded_tokenizer.texts_to_sequences(texts)
    # Pad to the same MAX_LEN used during training
    pads = pad_sequences(seqs, maxlen=meta['max_len'],
                         padding='post', truncating='post')
    # Run forward pass — output is sigmoid probability ∈ [0, 1]
    probs = loaded_model.predict(pads, verbose=0).flatten()
    print()
    for txt, prob in zip(texts, probs):
        label = 'Positive 🥰' if prob >= 0.5 else 'Negative 😤'
        print(f'  [{prob:.3f}]  {label:15s}  →  "{txt}"')


# ── Demo predictions ──────────────────────────────────────────
predict_sentiment([
    'Amazing product! Love it!',
    'Terrible, broke after 2 days.',
    'It is okay, nothing special.',
    'Absolutely the best purchase I have ever made!',
    'Complete waste of money, very disappointed.',
])
